# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mohidraheel/Machine-Learning-Practice/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# Week 4 — Baseline Action Score

**Lane:** CTR opportunity / content opportunity scoring  
**Decision period:** March 2026  
**Action:** Review pages with meaningful visibility whose CTR is below the normal CTR for their search-position range.

This notebook does three things:

1. Checks two signals used by the baseline rule.
2. Builds one transparent baseline score and ranked queue.
3. Reviews the top 10 recommendations with a skeptical view.

The baseline uses only information available during March 2026. It does not use future performance, future labels, or existing product flags when calculating the score.

## Setup

The data is loaded from FlyRank's anonymized warehouse using DuckDB.

The Hugging Face token is loaded from Colab Secrets. The token is never printed or saved in the notebook.

In [8]:
!pip -q install duckdb huggingface_hub pandas numpy

In [9]:
import os
import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

from google.colab import userdata
from huggingface_hub import login
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

print("Libraries loaded.")

Libraries loaded.


In [10]:
hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise ValueError(
        "HF_TOKEN was not found. Add HF_TOKEN in Colab Secrets "
        "and enable notebook access."
    )

os.environ["HF_TOKEN"] = hf_token

login(
    token=hf_token,
    add_to_git_credential=False,
)

print("Hugging Face authentication completed.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face authentication completed.


In [11]:
con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
con.execute("SET secret_directory='/tmp'")

con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    )
""")

WAREHOUSE_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/**/*.parquet"
)

con.execute(f"""
    CREATE OR REPLACE VIEW warehouse AS
    SELECT *
    FROM read_parquet(
        '{WAREHOUSE_PATH}',
        hive_partitioning=true
    )
""")

print("Warehouse view created.")

Warehouse view created.


In [17]:
march_count = con.execute("""
    SELECT COUNT(*) AS total_rows
    FROM warehouse
    WHERE CAST(month AS VARCHAR) = '2026-03'
""").df()

display(march_count)

assert int(march_count.loc[0, "total_rows"]) > 0, (
    "No March 2026 rows were found."
)

,total_rows
0,9841378


In [19]:
available_columns = schema_df["column_name"].tolist()

available_lookup = {
    column.lower(): column
    for column in available_columns
}


def pick_column(candidates, description):
    """
    Return the first warehouse column matching one of the candidates.
    Matching is case-insensitive.
    """
    for candidate in candidates:
        if candidate.lower() in available_lookup:
            return available_lookup[candidate.lower()]

    raise KeyError(
        f"Could not find a column for {description}.\n"
        f"Tried: {candidates}\n"
        f"Available columns: {available_columns}"
    )


def quote_identifier(column_name):
    """
    Safely quote a DuckDB column name.
    """
    return '"' + column_name.replace('"', '""') + '"'


DATE_COL = pick_column(
    [
        "report_date",
        "date",
        "performance_date",
        "event_date",
    ],
    "report date",
)

CLIENT_COL = pick_column(
    [
        "client_hash_id",
        "client_id",
        "account_id",
        "site_id",
    ],
    "anonymized client identifier",
)

CONTENT_COL = pick_column(
    [
        "content_hash_id",
        "content_id",
        "page_id",
        "document_id",
        "url_id",
    ],
    "anonymized content identifier",
)

IMPRESSIONS_COL = pick_column(
    [
        "gsc_impressions",
        "impressions",
        "search_impressions",
        "total_impressions",
    ],
    "impressions",
)

CLICKS_COL = pick_column(
    [
        "gsc_clicks",
        "clicks",
        "search_clicks",
        "total_clicks",
    ],
    "clicks",
)

POSITION_COL = pick_column(
    [
        "gsc_avg_position",
        "avg_position",
        "average_position",
        "position",
        "weighted_position",
    ],
    "average search position",
)

GSC_AVAILABLE_COL = pick_column(
    [
        "gsc_data_available",
    ],
    "GSC availability flag",
)

print("Resolved columns")
print("----------------")
print("Date:", DATE_COL)
print("Client ID:", CLIENT_COL)
print("Content ID:", CONTENT_COL)
print("Impressions:", IMPRESSIONS_COL)
print("Clicks:", CLICKS_COL)
print("Position:", POSITION_COL)
print("GSC available:", GSC_AVAILABLE_COL)

Resolved columns
----------------
Date: report_date
Client ID: client_hash_id
Content ID: content_hash_id
Impressions: gsc_impressions
Clicks: gsc_clicks
Position: gsc_avg_position
GSC available: gsc_data_available


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

### Rule in plain words

I compare each page with pages in a similar search-position range. A page becomes a review candidate when:

1. it received at least 100 observed impressions during March 2026; and
2. its observed CTR is below the median CTR of its search-position bucket.

The score is higher when the page has more impressions and a larger CTR gap. This is a **decision-support** rule. It does not prove that a page is stale or that changing it will improve performance.

### Reason code and action

- **Reason code:** `HIGH_VISIBILITY_CTR_GAP`
- **Action label:** `REVIEW_CTR_OPPORTUNITY`

The reason code means the page has measured visibility and an observed CTR below the median for comparable search positions. The action asks for human review of the title, snippet, intent match, and content freshness.

In [20]:
client_sql = quote_identifier(CLIENT_COL)
content_sql = quote_identifier(CONTENT_COL)
impressions_sql = quote_identifier(IMPRESSIONS_COL)
clicks_sql = quote_identifier(CLICKS_COL)
position_sql = quote_identifier(POSITION_COL)

page_frame_sql = f"""
SELECT
    {client_sql} AS client_id,
    {content_sql} AS content_id,

    SUM(COALESCE({impressions_sql}, 0)) AS impressions_31d,
    SUM(COALESCE({clicks_sql}, 0)) AS clicks_31d,

    CASE
        WHEN SUM(COALESCE({impressions_sql}, 0)) > 0
        THEN
            SUM(COALESCE({clicks_sql}, 0)) * 1.0
            / SUM(COALESCE({impressions_sql}, 0))
        ELSE NULL
    END AS ctr_31d,

    CASE
        WHEN SUM(
            CASE
                WHEN {position_sql} IS NOT NULL
                THEN COALESCE({impressions_sql}, 0)
                ELSE 0
            END
        ) > 0
        THEN
            SUM(
                CASE
                    WHEN {position_sql} IS NOT NULL
                    THEN {position_sql} * COALESCE({impressions_sql}, 0)
                    ELSE 0
                END
            ) * 1.0
            /
            SUM(
                CASE
                    WHEN {position_sql} IS NOT NULL
                    THEN COALESCE({impressions_sql}, 0)
                    ELSE 0
                END
            )
        ELSE NULL
    END AS weighted_position_31d,

    COUNT(*) AS source_rows

FROM warehouse

WHERE CAST(month AS VARCHAR) = '2026-03'

GROUP BY
    {client_sql},
    {content_sql}

HAVING
    SUM(COALESCE({impressions_sql}, 0)) >= 100
"""

frame = con.execute(page_frame_sql).df()

print("Eligible page rows:", len(frame))
display(frame.head(10))

assert len(frame) > 0, "The March page-level frame is empty."
assert frame["client_id"].notna().all()
assert frame["content_id"].notna().all()
assert frame["impressions_31d"].ge(100).all()
assert frame["clicks_31d"].ge(0).all()
assert frame["ctr_31d"].dropna().between(0, 1).all()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible page rows: 101441


,client_id,content_id,impressions_31d,clicks_31d,ctr_31d,weighted_position_31d,source_rows
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,0.001754,4.450877,31
1,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,0.000000,5.637584,31
2,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,0.004222,6.906404,31
3,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,0.005776,3.950542,31
4,client_73cda7b4e4f265ea,content_2662845f598544ef,150.0,1.0,0.006667,7.506667,31
5,client_73cda7b4e4f265ea,content_712c365258cee05c,6048.0,23.0,0.003803,4.931878,31
6,client_73cda7b4e4f265ea,content_476c37c366920c1b,223.0,0.0,0.000000,61.538117,31
7,client_73cda7b4e4f265ea,content_3dba50ae010f3f30,357.0,1.0,0.002801,18.876751,31
8,client_73cda7b4e4f265ea,content_d720dde3701523c0,132.0,1.0,0.007576,30.863636,31
9,client_73cda7b4e4f265ea,content_098eedbd77ec1de1,281.0,1.0,0.003559,38.199288,31


### Position-aware CTR reference

Search position affects expected CTR, so I do not apply one universal CTR threshold. I place pages into broad position buckets and use the observed median CTR in each bucket as the reference value. The table prints `n` so small buckets are visible.

In [21]:
baseline = frame.dropna(
    subset=[
        "impressions_31d",
        "ctr_31d",
        "weighted_position_31d",
    ]
).copy()

position_bins = [
    0,
    3,
    5,
    10,
    20,
    50,
    float("inf"),
]

position_labels = [
    "1-3",
    "4-5",
    "6-10",
    "11-20",
    "21-50",
    "51+",
]

baseline["position_bucket"] = pd.cut(
    baseline["weighted_position_31d"],
    bins=position_bins,
    labels=position_labels,
    include_lowest=True,
)

expected_ctr_table = (
    baseline
    .groupby("position_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        expected_ctr=("ctr_31d", "median"),
        mean_ctr=("ctr_31d", "mean"),
        median_impressions=("impressions_31d", "median"),
    )
    .reset_index()
)

expected_ctr_table["expected_ctr_pct"] = (
    expected_ctr_table["expected_ctr"] * 100
).round(2)

expected_ctr_table["mean_ctr_pct"] = (
    expected_ctr_table["mean_ctr"] * 100
).round(2)

display(
    expected_ctr_table[
        [
            "position_bucket",
            "n",
            "expected_ctr_pct",
            "mean_ctr_pct",
            "median_impressions",
        ]
    ]
)

baseline = baseline.merge(
    expected_ctr_table[
        ["position_bucket", "n", "expected_ctr"]
    ].rename(columns={"n": "bucket_n"}),
    on="position_bucket",
    how="left",
)

baseline["ctr_gap"] = (
    baseline["expected_ctr"] - baseline["ctr_31d"]
).clip(lower=0)

baseline["ctr_gap_pct_points"] = (
    baseline["ctr_gap"] * 100
).round(2)

assert baseline["expected_ctr"].notna().all()
print("Position-aware CTR reference created.")

,position_bucket,n,expected_ctr_pct,mean_ctr_pct,median_impressions
0,1-3,10194,0.21,0.34,1594.0
1,4-5,17167,0.23,0.36,1558.0
2,6-10,30644,0.16,0.30,807.0
3,11-20,19547,0.10,0.24,564.0
4,21-50,19758,0.00,0.14,571.0
5,51+,4131,0.00,0.05,201.0


Position-aware CTR reference created.


**Observed conclusion:** The table provides a measured, position-aware CTR reference. The baseline uses this relationship directionally; it does not assume that every difference is caused by content quality.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The score combines:

- **60% impression-volume percentile:** the measured size of the opportunity;
- **40% CTR-gap percentile:** the measured distance below the position-bucket median.

Only pages with a positive CTR gap enter the queue. The rule outputs one reason code and one action label.

In [22]:
baseline["volume_percentile"] = (
    baseline["impressions_31d"]
    .rank(method="average", pct=True)
)

baseline["ctr_gap_percentile"] = (
    baseline["ctr_gap"]
    .rank(method="average", pct=True)
)

baseline["baseline_action_score"] = (
    100
    * (
        0.60 * baseline["volume_percentile"]
        + 0.40 * baseline["ctr_gap_percentile"]
    )
).round(2)

queue = baseline[
    baseline["ctr_gap"] > 0
].copy()

queue["reason_code"] = "HIGH_VISIBILITY_CTR_GAP"
queue["action_label"] = "REVIEW_CTR_OPPORTUNITY"

queue = queue.sort_values(
    by=[
        "baseline_action_score",
        "impressions_31d",
        "ctr_gap",
    ],
    ascending=[
        False,
        False,
        False,
    ],
).reset_index(drop=True)

queue.insert(
    0,
    "rank",
    range(1, len(queue) + 1),
)

queue_columns = [
    "rank",
    "client_id",
    "content_id",
    "baseline_action_score",
    "reason_code",
    "action_label",
    "impressions_31d",
    "clicks_31d",
    "ctr_31d",
    "weighted_position_31d",
    "position_bucket",
    "expected_ctr",
    "ctr_gap",
    "ctr_gap_pct_points",
    "volume_percentile",
    "ctr_gap_percentile",
    "bucket_n",
]

queue = queue[queue_columns]

print("Eligible pages:", len(frame))
print("Pages entering the ranked queue:", len(queue))
display(queue.head(20))

assert len(queue) > 0, "No pages entered the ranked queue."
assert queue["baseline_action_score"].between(0, 100).all()
assert queue["baseline_action_score"].is_monotonic_decreasing
assert queue["reason_code"].nunique() == 1
assert queue["action_label"].nunique() == 1

Eligible pages: 101441
Pages entering the ranked queue: 38768


,rank,client_id,content_id,baseline_action_score,reason_code,action_label,impressions_31d,clicks_31d,ctr_31d,weighted_position_31d,position_bucket,expected_ctr,ctr_gap,ctr_gap_pct_points,volume_percentile,ctr_gap_percentile,bucket_n
0,1,client_62f4a7e64f5e0096,content_0c5606abaaab3178,98.99,HIGH_VISIBILITY_CTR_GAP,REVIEW_CTR_OPPORTUNITY,38865.0,0.0,0.000000,4.804554,4-5,0.002309,0.002309,0.23,0.994391,0.983123,17167
1,2,client_e547b89c05043229,content_713b157e9c77690a,98.49,HIGH_VISIBILITY_CTR_GAP,REVIEW_CTR_OPPORTUNITY,24908.0,0.0,0.000000,3.431106,4-5,0.002309,0.002309,0.23,0.986140,0.983123,17167
2,3,client_62f4a7e64f5e0096,content_37a6fac676c8cebb,98.43,HIGH_VISIBILITY_CTR_GAP,REVIEW_CTR_OPPORTUNITY,48049.0,4.0,0.000083,4.231056,4-5,0.002309,0.002226,0.22,0.996441,0.966187,17167
3,4,client_62f4a7e64f5e0096,content_8d395745c4d76f9e,97.99,HIGH_VISIBILITY_CTR_GAP,REVIEW_CTR_OPPORTUNITY,28250.0,4.0,0.000142,4.531646,4-5,0.002309,0.002168,0.22,0.989196,0.966059,17167
4,5,client_62f4a7e64f5e0096,content_a7b383d517ba8a7c,97.95,HIGH_VISIBILITY_CTR_GAP,REVIEW_CTR_OPPORTUNITY,27431.0,4.0,0.000146,3.338887,4-5,0.002309,0.002164,0.22,0.988432,0.966039,17167
5,6,client_23a62021009f63c4,content_bf078007df823490,97.93,HIGH_VISIBILITY_CTR_GAP,REVIEW_CTR_OPPORTUNITY,44707.0,0.0,0.000000,1.400049,1-3,0.002114,0.002114,0.21,0.995870,0.954348,10194
6,7,client_23a62021009f63c4,content_fa4cf3aa5ce67bb8,97.86,HIGH_VISIBILITY_CTR_GAP,REVIEW_CTR_OPPORTUNITY,25588.0,1.0,0.000039,4.832812,4-5,0.002309,0.002270,0.23,0.986810,0.966237,17167
7,8,client_73cda7b4e4f265ea,content_345c8feeab5d080a,97.73,HIGH_VISIBILITY_CTR_GAP,REVIEW_CTR_OPPORTUNITY,23418.0,1.0,0.000043,4.969980,4-5,0.002309,0.002267,0.23,0.984602,0.966227,17167
8,9,client_73cda7b4e4f265ea,content_8e1334d6356668e3,97.70,HIGH_VISIBILITY_CTR_GAP,REVIEW_CTR_OPPORTUNITY,134984.0,1.0,0.000007,2.693038,1-3,0.002114,0.002107,0.21,0.999763,0.942784,10194
9,10,client_73cda7b4e4f265ea,content_fec55986a1868d62,97.69,HIGH_VISIBILITY_CTR_GAP,REVIEW_CTR_OPPORTUNITY,124075.0,1.0,0.000008,0.308426,1-3,0.002114,0.002106,0.21,0.999704,0.942775,10194


In [23]:
OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = OUTPUT_DIR / "baseline_action_score.csv"

queue.to_csv(
    csv_path,
    index=False,
)

metrics = {
    "assignment": "ML-07 baseline action score",
    "lane": "CTR opportunity / content opportunity scoring",
    "data_period": "2026-03",
    "minimum_impressions": 100,
    "eligible_page_rows": int(len(frame)),
    "queue_rows": int(len(queue)),
    "score_name": "baseline_action_score",
    "reason_code": "HIGH_VISIBILITY_CTR_GAP",
    "action_label": "REVIEW_CTR_OPPORTUNITY",
    "volume_weight": 0.60,
    "ctr_gap_weight": 0.40,
    "future_inputs_used": False,
    "label_derived_inputs_used": False,
    "product_flags_used": False,
}

metrics_path = OUTPUT_DIR / "w04_baseline_metrics.json"

with open(metrics_path, "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2)

print("CSV written to:", csv_path)
print("Metrics JSON written to:", metrics_path)
print("CSV rows:", len(queue))

assert csv_path.exists()
assert metrics_path.exists()

display(pd.Series(metrics, name="value").to_frame())

CSV written to: work/outputs/baseline_action_score.csv
Metrics JSON written to: work/outputs/w04_baseline_metrics.json
CSV rows: 38768


,value
assignment,ML-07 baseline action score
lane,CTR opportunity / content opportunity scoring
data_period,2026-03
minimum_impressions,100
eligible_page_rows,101441
queue_rows,38768
score_name,baseline_action_score
reason_code,HIGH_VISIBILITY_CTR_GAP
action_label,REVIEW_CTR_OPPORTUNITY
volume_weight,0.6


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

For each top-20 row, I show:

- the decision-support action;
- the reason code;
- a confidence note based only on the two measured score components;
- why the row is present;
- what could make the recommendation wrong.

Confidence is directional, not a probability.

In [24]:
top20 = queue.head(20).copy()

top20["actual_ctr_pct"] = (
    top20["ctr_31d"] * 100
).round(2)

top20["expected_ctr_pct"] = (
    top20["expected_ctr"] * 100
).round(2)


def confidence_note(row):
    """Create a transparent directional confidence note."""
    if (
        row["volume_percentile"] >= 0.80
        and row["ctr_gap_percentile"] >= 0.80
    ):
        return (
            "Higher directional confidence: both measured signals "
            "are in the top 20% of eligible pages."
        )

    if (
        row["volume_percentile"] >= 0.60
        and row["ctr_gap_percentile"] >= 0.60
    ):
        return (
            "Medium directional confidence: both measured signals "
            "are above the middle of the eligible-page distribution."
        )

    return (
        "Lower directional confidence: one measured signal is not "
        "especially strong."
    )


top20["action"] = (
    "Review title, search snippet, intent match, and content freshness."
)

top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1,
)

top20["why_it_is_here"] = top20.apply(
    lambda row: (
        f"Ranked #{int(row['rank'])}: "
        f"{int(row['impressions_31d']):,} observed impressions and "
        f"CTR {row['ctr_gap_pct_points']:.2f} percentage points below "
        f"the median for position bucket {row['position_bucket']}."
    ),
    axis=1,
)

top20["what_would_make_it_wrong"] = (
    "The recommendation could be wrong if low CTR is normal for the "
    "query intent, caused by SERP features, branded competition or "
    "seasonality, measured in a bucket that is too broad, or attached "
    "to a page that is already current."
)

top20_review_columns = [
    "rank",
    "client_id",
    "content_id",
    "action",
    "reason_code",
    "confidence_note",
    "why_it_is_here",
    "what_would_make_it_wrong",
]

display(top20[top20_review_columns])

assert len(top20) == min(20, len(queue))
assert top20["action"].notna().all()
assert top20["reason_code"].notna().all()
assert top20["confidence_note"].notna().all()
assert top20["what_would_make_it_wrong"].notna().all()

,rank,client_id,content_id,action,reason_code,confidence_note,why_it_is_here,what_would_make_it_wrong
0,1,client_62f4a7e64f5e0096,content_0c5606abaaab3178,"Review title, search snippet, intent match, and content freshness.",HIGH_VISIBILITY_CTR_GAP,Higher directional confidence: both measured signals are in the top 20% of eligible pages.,"Ranked #1: 38,865 observed impressions and CTR 0.23 percentage points below the median for position bucket 4-5.","The recommendation could be wrong if low CTR is normal for the query intent, caused by SERP features, branded competition or seasonality, measured in a bucket that is too broad, or attached to a p..."
1,2,client_e547b89c05043229,content_713b157e9c77690a,"Review title, search snippet, intent match, and content freshness.",HIGH_VISIBILITY_CTR_GAP,Higher directional confidence: both measured signals are in the top 20% of eligible pages.,"Ranked #2: 24,908 observed impressions and CTR 0.23 percentage points below the median for position bucket 4-5.","The recommendation could be wrong if low CTR is normal for the query intent, caused by SERP features, branded competition or seasonality, measured in a bucket that is too broad, or attached to a p..."
2,3,client_62f4a7e64f5e0096,content_37a6fac676c8cebb,"Review title, search snippet, intent match, and content freshness.",HIGH_VISIBILITY_CTR_GAP,Higher directional confidence: both measured signals are in the top 20% of eligible pages.,"Ranked #3: 48,049 observed impressions and CTR 0.22 percentage points below the median for position bucket 4-5.","The recommendation could be wrong if low CTR is normal for the query intent, caused by SERP features, branded competition or seasonality, measured in a bucket that is too broad, or attached to a p..."
3,4,client_62f4a7e64f5e0096,content_8d395745c4d76f9e,"Review title, search snippet, intent match, and content freshness.",HIGH_VISIBILITY_CTR_GAP,Higher directional confidence: both measured signals are in the top 20% of eligible pages.,"Ranked #4: 28,250 observed impressions and CTR 0.22 percentage points below the median for position bucket 4-5.","The recommendation could be wrong if low CTR is normal for the query intent, caused by SERP features, branded competition or seasonality, measured in a bucket that is too broad, or attached to a p..."
4,5,client_62f4a7e64f5e0096,content_a7b383d517ba8a7c,"Review title, search snippet, intent match, and content freshness.",HIGH_VISIBILITY_CTR_GAP,Higher directional confidence: both measured signals are in the top 20% of eligible pages.,"Ranked #5: 27,431 observed impressions and CTR 0.22 percentage points below the median for position bucket 4-5.","The recommendation could be wrong if low CTR is normal for the query intent, caused by SERP features, branded competition or seasonality, measured in a bucket that is too broad, or attached to a p..."
5,6,client_23a62021009f63c4,content_bf078007df823490,"Review title, search snippet, intent match, and content freshness.",HIGH_VISIBILITY_CTR_GAP,Higher directional confidence: both measured signals are in the top 20% of eligible pages.,"Ranked #6: 44,707 observed impressions and CTR 0.21 percentage points below the median for position bucket 1-3.","The recommendation could be wrong if low CTR is normal for the query intent, caused by SERP features, branded competition or seasonality, measured in a bucket that is too broad, or attached to a p..."
6,7,client_23a62021009f63c4,content_fa4cf3aa5ce67bb8,"Review title, search snippet, intent match, and content freshness.",HIGH_VISIBILITY_CTR_GAP,Higher directional confidence: both measured signals are in the top 20% of eligible pages.,"Ranked #7: 25,588 observed impressions and CTR 0.23 percentage points below the median for position bucket 4-5.","The recommendation could be wrong if low CTR is normal for the query intent, caused by SERP features, branded competition or seasonality, measured in a bucket that is too broad, or attached to a p..."
7,8,client_73cda7b4e4f265ea

### Top-20 review conclusion

The top-ranked rows combine substantial observed visibility with CTR below the median for comparable positions. That is enough to justify review, but not enough to diagnose the cause.

Possible explanations include search intent, SERP features, query mix, seasonality, strong branded results, a weak title rather than weak content, an already-current page, broad position buckets, or ordinary measurement noise. The queue should guide a person, not automatically trigger an edit.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks are queue rows where at least one score component is below the 60th percentile. They entered the queue because their CTR gap is positive, but their evidence is less balanced.

The leakage check attacks the rule directly. It confirms that the score uses no future windows, target or label fields, product-generated flags, client names, URLs, or private search queries.

In [25]:
weak_picks = queue[
    (queue["volume_percentile"] < 0.60)
    | (queue["ctr_gap_percentile"] < 0.60)
].head(10).copy()

weak_picks["actual_ctr_pct"] = (
    weak_picks["ctr_31d"] * 100
).round(2)

weak_picks["expected_ctr_pct"] = (
    weak_picks["expected_ctr"] * 100
).round(2)


def weak_reason(row):
    reasons = []

    if row["volume_percentile"] < 0.60:
        reasons.append("impression opportunity is below the 60th percentile")

    if row["ctr_gap_percentile"] < 0.60:
        reasons.append("CTR gap is below the 60th percentile")

    return "; ".join(reasons)


weak_picks["why_it_looks_weak"] = weak_picks.apply(
    weak_reason,
    axis=1,
)

weak_picks["what_to_check"] = (
    "Check search intent, SERP features, bucket fit, page freshness, "
    "and whether a small change in clicks would remove the observed gap."
)

display(
    weak_picks[
        [
            "rank",
            "client_id",
            "content_id",
            "baseline_action_score",
            "impressions_31d",
            "position_bucket",
            "actual_ctr_pct",
            "expected_ctr_pct",
            "ctr_gap_pct_points",
            "why_it_looks_weak",
            "what_to_check",
        ]
    ]
)

,rank,client_id,content_id,baseline_action_score,impressions_31d,position_bucket,actual_ctr_pct,expected_ctr_pct,ctr_gap_pct_points,why_it_looks_weak,what_to_check
8175,8176,client_62f4a7e64f5e0096,content_6252b366805e5ebe,75.32,1215.0,4-5,0.0,0.23,0.23,impression opportunity is below the 60th percentile,"Check search intent, SERP features, bucket fit, page freshness, and whether a small change in clicks would remove the observed gap."
8176,8177,client_a80fca3f171ed1de,content_648a98629639ba30,75.32,1215.0,4-5,0.0,0.23,0.23,impression opportunity is below the 60th percentile,"Check search intent, SERP features, bucket fit, page freshness, and whether a small change in clicks would remove the observed gap."
8189,8190,client_e5c2aa26a8598242,content_f2f658c7b4076d5c,75.29,1212.0,4-5,0.0,0.23,0.23,impression opportunity is below the 60th percentile,"Check search intent, SERP features, bucket fit, page freshness, and whether a small change in clicks would remove the observed gap."
8190,8191,client_73cda7b4e4f265ea,content_adc2004821136a9f,75.29,1212.0,4-5,0.0,0.23,0.23,impression opportunity is below the 60th percentile,"Check search intent, SERP features, bucket fit, page freshness, and whether a small change in clicks would remove the observed gap."
8221,8222,client_62f4a7e64f5e0096,content_ddf34a811746c109,75.22,1207.0,4-5,0.0,0.23,0.23,impression opportunity is below the 60th percentile,"Check search intent, SERP features, bucket fit, page freshness, and whether a small change in clicks would remove the observed gap."
8240,8241,client_400c21c81c8b46ef,content_d08b9f46b11c46b3,75.19,1204.0,4-5,0.0,0.23,0.23,impression opportunity is below the 60th percentile,"Check search intent, SERP features, bucket fit, page freshness, and whether a small change in clicks would remove the observed gap."
8256,8257,client_73cda7b4e4f265ea,content_9b040f4248a7099c,75.17,1202.0,4-5,0.0,0.23,0.23,impression opportunity is below the 60th percentile,"Check search intent, SERP features, bucket fit, page freshness, and whether a small change in clicks would remove the observed gap."
8295,8296,client_73cda7b4e4f265ea,content_0c22696faca56ae3,75.10,1197.0,4-5,0.0,0.23,0.23,impression opportunity is below the 60th percentile,"Check search intent, SERP features, bucket fit, page freshness, and whether a small change in clicks would remove the observed gap."
8300,8301,client_73cda7b4e4f265ea,content_d25a38c2f8034764,75.09,1196.0,4-5,0.0,0.23,0.23,impression opportunity is below the 60th percentile,"Check search intent, SERP features, bucket fit, page freshness, and whether a small change in clicks would remove the observed gap."
8301,8302,client_e547b89c05043229,content_a9d7079455d529fb,75.09,1196.0,4-5,0.0,0.23,0.23,impression opportunity is below the 60th percentile,"Check search intent, SERP features, bucket fit, page freshness, and whether a small change in clicks would remove the observed gap."


### Weak-pick conclusion

These picks are more likely to be wrong when the impression opportunity is limited, the CTR gap is small, a few clicks would materially change the result, or the position bucket hides different query intents. They should remain below the strongest review candidates.

In [26]:
score_input_columns = [
    "impressions_31d",
    "ctr_31d",
    "weighted_position_31d",
    "position_bucket",
    "expected_ctr",
    "ctr_gap",
    "volume_percentile",
    "ctr_gap_percentile",
]

forbidden_terms = [
    "future",
    "next_",
    "label",
    "target",
    "outcome",
    "after",
    "refresh_flag",
    "quick_win_flag",
    "product_flag",
    "recommendation_flag",
]

leakage_hits = [
    column
    for column in score_input_columns
    if any(
        term in column.lower()
        for term in forbidden_terms
    )
]

print("Score inputs:")
for column in score_input_columns:
    print("-", column)

print("\nPotential leakage hits:", leakage_hits)

assert leakage_hits == [], (
    f"Potential future, label, or product-flag leakage: {leakage_hits}"
)

print("\nPASS: No future, label-derived, or product-flag fields were used.")

Score inputs:
- impressions_31d
- ctr_31d
- weighted_position_31d
- position_bucket
- expected_ctr
- ctr_gap
- volume_percentile
- ctr_gap_percentile

Potential leakage hits: []

PASS: No future, label-derived, or product-flag fields were used.


In [27]:
private_terms = [
    "client_name",
    "company_name",
    "domain",
    "url",
    "query",
    "keyword",
    "page_title",
    "private",
]

private_output_hits = [
    column
    for column in queue.columns
    if any(
        term in column.lower()
        for term in private_terms
    )
]

print("Potential private output columns:", private_output_hits)

assert private_output_hits == [], (
    f"Potential private output fields found: {private_output_hits}"
)

assert set(score_input_columns).issubset(
    set(baseline.columns)
), "A declared score input is missing."

assert "month" not in score_input_columns
assert queue["reason_code"].eq("HIGH_VISIBILITY_CTR_GAP").all()
assert queue["action_label"].eq("REVIEW_CTR_OPPORTUNITY").all()
assert csv_path.exists()
assert metrics_path.exists()

print("PASS: Privacy, leakage, rule, and output checks completed.")

Potential private output columns: []
PASS: Privacy, leakage, rule, and output checks completed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [28]:
print("ML-07 / Week 4 notebook complete")
print("--------------------------------")
print("Eligible pages:", len(frame))
print("Ranked queue rows:", len(queue))
print("Top rows reviewed:", len(top20))
print("CSV:", csv_path)
print("Metrics:", metrics_path)

ML-07 / Week 4 notebook complete
--------------------------------
Eligible pages: 101441
Ranked queue rows: 38768
Top rows reviewed: 20
CSV: work/outputs/baseline_action_score.csv
Metrics: work/outputs/w04_baseline_metrics.json
